Objectif : 

L'objectif de ce notebook est de développer un système de recommandation basé sur le filtrage collaboratif. Contrairement au modèle content-based, cette approche utilise les notes attribuées par les utilisateurs afin de mesurer la similarité entre les films et générer des recommandations personnalisées.


I. Chargement des données

In [1]:
import pandas as pd
import numpy as np

films = pd.read_csv('/Users/nazmanazirhussain/Desktop/RecommandationFilm/data/nettoye/movies_clean.csv')
notes = pd.read_csv('/Users/nazmanazirhussain/Desktop/RecommandationFilm/data/nettoye/ratings_clean.csv')

print(films.shape)
print(notes.shape)
notes.head()

(9742, 6)
(100836, 5)


,userId,movieId,rating,timestamp,date
0,1,1,4.0,964982703,2000-07-30 18:45:03
1,1,3,4.0,964981247,2000-07-30 18:20:47
2,1,6,4.0,964982224,2000-07-30 18:37:04
3,1,47,5.0,964983815,2000-07-30 19:03:35
4,1,50,5.0,964982931,2000-07-30 18:48:51


II. Construction de la matrice utilisateur-film

C'est le coeur du filtrage collaboratif. Il y aura un grand tableau où chaque ligne est un utilisateur, chaque colonne est un film. Chaque case contient la note donnée (ou vide si l'utilisateur n'a pas noté ce film). 

In [2]:
# On transforme le tableau "long" (une ligne par note) 
# en un tableau "large" (une ligne par utilisateur, une colonne par film)
matrice_utilisateur_film = notes.pivot_table(index='userId', columns='movieId', values='rating')

print("Dimensions :", matrice_utilisateur_film.shape)
matrice_utilisateur_film.head()

Dimensions : (610, 9724)


movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


La matrice contient 610 utilisateurs et 9724 films, soit 610 × 9724 combinaisons possibles. Chaque ligne représente un utilisateur et chaque colonne un film. Les cellules contiennent la note attribuée lorsque celle-ci existe.

Commentaire : Pour mettre en place le filtrage collaboratif, nous avons transformé notre tableau de notes (ratings_clean.csv), initialement structuré sous une forme dite "longue" (une ligne par note, avec les colonnes userId, movieId, rating), en une matrice dite "large" grâce à un pivot de tableau (pivot_table). Cette nouvelle matrice place chaque utilisateur en ligne et chaque film en colonne, chaque case contenant la note attribuée par cet utilisateur à ce film. Cette structure est indispensable pour le filtrage collaboratif, car elle permet de comparer directement les comportements de notation entre films ou entre utilisateurs, ce qui n'est pas possible avec le format initial en liste. La grande majorité des cases de cette matrice sont vides (NaN), puisqu'un utilisateur ne note en pratique qu'une infime partie des films disponibles dans le catalogue. C'est ce qu'on appelle une matrice creuse (sparse matrix), un phénomène caractéristique et bien documenté dans la littérature des systèmes de recommandation.

Vérifier le taux de remplissage 

In [3]:
nb_notes_possibles = matrice_utilisateur_film.shape[0] * matrice_utilisateur_film.shape[1]
nb_notes_reelles = notes.shape[0]
taux_remplissage = nb_notes_reelles / nb_notes_possibles * 100

print(f"Taux de remplissage de la matrice : {taux_remplissage:.2f}%")

Taux de remplissage de la matrice : 1.70%


Dans notre cas, seulement 1,70% des cases possibles de cette matrice contiennent une véritable note, ce qui confirme ce phénomène de rareté des données (sparsity) : plus de 98% de la matrice est constituée de valeurs manquantes. Ce constat illustre concrètement la difficulté du problème de recommandation par filtrage collaboratif, puisque le modèle doit être capable d'estimer des préférences à partir d'un signal extrêmement limité. Cette rareté justifie également certains choix méthodologiques ultérieurs, comme le remplacement des valeurs manquantes par 0 avant le calcul de similarité, une approximation qui sera discutée comme limite du modèle.

Gestion des valeurs manquantes

Les valeurs manquantes correspondent à des films qui n'ont jamais été notés par certains utilisateurs. Elles ne représentent pas une note égale à zéro mais une absence d'information. Afin de calculer la similarité cosinus, ces valeurs sont temporairement remplacées par 0.

In [4]:
matrice_remplie = matrice_utilisateur_film.fillna(0)
matrice_remplie.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


III. Calcul de la similarité entre films (item-based)

Nous avons choisi une approche item-based plutôt que user-based car les similarités entre les films sont plus stables que les similarités entre utilisateurs. Cette approche est également plus adaptée lorsque le nombre d'utilisateurs est important.

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

# On veut comparer les FILMS entre eux, donc on transpose 
# (les films deviennent les lignes, les utilisateurs deviennent les colonnes)
similarite_films = cosine_similarity(matrice_remplie.T)

print("Dimensions de la matrice de similarité entre films :", similarite_films.shape)

Dimensions de la matrice de similarité entre films : (9724, 9724)


Le dataset "movies_clean" contient 9742 films mais ici on a seulement 9724. Cela signifie que le reste des films n'ont pas été noté par les utilisateurs. 

In [6]:
# Vérification 

print("Films dans le catalogue :", films['movieId'].nunique())
print("Films avec au moins une note :", notes['movieId'].nunique())

Films dans le catalogue : 9742
Films avec au moins une note : 9724


IV. Création de la fonction de recommandation


In [7]:
def films_similaires_collaboratif(id_film, n=10):
    # Trouver la position du film dans la matrice
    idx = list(matrice_remplie.columns).index(id_film)
    
    # Récupérer les scores de similarité avec tous les autres films
    scores = list(enumerate(similarite_films[idx]))
    
    # Trier du plus similaire au moins similaire (on exclut le film lui-même)
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:n+1]
    
    # Récupérer les movieId correspondants
    ids_films_similaires = [matrice_remplie.columns[i[0]] for i in scores]
    
    return films[films['movieId'].isin(ids_films_similaires)][['titre', 'genres', 'annee']]

In [8]:
# Teste avec le film Toy Story, movieId = 1
films_similaires_collaboratif(1)


,titre,genres,annee
224,Star Wars: Episode IV - A New Hope,Action|Adventure|Sci-Fi,1977
314,Forrest Gump,Comedy|Drama|Romance|War,1994
322,"Lion King, The",Adventure|Animation|Children|Drama|Musical|IMAX,1994
418,Jurassic Park,Action|Adventure|Sci-Fi|Thriller,1993
546,Mission: Impossible,Action|Adventure|Mystery|Thriller,1996
615,Independence Day (a.k.a. ID4),Action|Adventure|Sci-Fi|Thriller,1996
911,Star Wars: Episode VI - Return of the Jedi,Action|Adventure|Sci-Fi,1983
964,Groundhog Day,Comedy|Fantasy|Romance,1993
969,Back to the Future,Adventure|Comedy|Sci-Fi,1985
2355,Toy Story 2,Adventure|Animation|Children|Comedy|Fantasy,1999


Commentaire : Afin d'évaluer qualitativement nos deux premiers modèles, nous avons comparé leurs recommandations respectives pour un même film de référence, Toy Story. Le modèle content-based, basé uniquement sur les genres, recommande exclusivement des films d'animation destinés à un public familial (Antz, Monsters, Inc., Shrek the Third), partageant tous une combinaison de genres quasiment identique. Le modèle collaboratif, à l'inverse, recommande des films de genres très variés (science-fiction, comédie, action, drame), tels que Star Wars, Forrest Gump ou Jurassic Park. Ces films n'ont a priori aucun lien thématique avec Toy Story, mais ont été notés de manière similaire par les mêmes utilisateurs, ce qui suggère qu'ils s'adressent à un public partageant des habitudes de visionnage communes, notamment de grands succès populaires des années 1990.
Cette différence illustre la complémentarité des deux approches : le content-based capture une similarité objective, fondée sur les caractéristiques intrinsèques des films, tandis que le filtrage collaboratif capture une similarité comportementale, fondée sur les habitudes réelles des utilisateurs. Cette observation justifie l'intérêt, dans la suite du projet, d'envisager une approche hybride combinant les deux méthodes afin de proposer des recommandations à la fois pertinentes sur le plan du contenu et representatives des goûts effectifs des utilisateurs.

In [9]:
# Teste avec le film Pulp Fiction, movieId = 296
films_similaires_collaboratif(296)

,titre,genres,annee
43,Seven (a.k.a. Se7en),Mystery|Thriller,1995
46,"Usual Suspects, The",Crime|Mystery|Thriller,1995
97,Braveheart,Action|Drama|War,1995
277,"Shawshank Redemption, The",Crime|Drama,1994
314,Forrest Gump,Comedy|Drama|Romance|War,1994
507,Terminator 2: Judgment Day,Action|Sci-Fi,1991
510,"Silence of the Lambs, The",Crime|Horror|Thriller,1991
520,Fargo,Comedy|Crime|Drama|Thriller,1996
828,Reservoir Dogs,Crime|Mystery|Thriller,1992
2226,Fight Club,Action|Crime|Drama|Thriller,1999


Commentaire : Un second test avec Pulp Fiction confirme et nuance nos observations précédentes. Contrairement au cas de Toy Story, les deux modèles convergent ici vers un univers thématique proche (crime, thriller), mais avec une précision différente. Le content-based recommande des films partageant exactement la même combinaison de genres (Fargo, In Bruges), tandis que le filtrage collaboratif propose des films cultes du même esprit cinématographique (Fight Club, Seven, Reservoir Dogs, The Usual Suspects), reconnus par la critique comme thématiquement et stylistiquement proches de Pulp Fiction, sans pour autant partager une étiquette de genre strictement identique.
Ce second exemple montre que la pertinence du filtrage collaboratif dépend fortement du volume et de la cohérence des comportements de notation disponibles pour le film de référence. Il confirme également l'intérêt d'une approche hybride, qui pourrait combiner la garantie thématique du content-based avec la finesse de goût capturée par le filtrage collaboratif.

V. Évaluation des modèles de recommandation : RMSE et Precision@k / Recall@k

Pour évaluer objectivement la qualité de nos modèles, au-delà des observations qualitatives précédentes, nous utilisons deux familles de métriques qui répondent à des questions différentes.

Le RMSE (Root Mean Squared Error, ou racine de l'erreur quadratique moyenne) mesure la précision d'un modèle à prédire une note. Concrètement, on cache une partie des notes réelles du dataset, on demande au modèle de les deviner, puis on compare la note prédite à la note réelle. Plus l'écart moyen entre les notes prédites et les notes réelles est faible, plus le RMSE est bas, et meilleur est le modèle. Cette métrique répond à la question : "À quel point mon modèle devine-t-il correctement la note qu'un utilisateur donnerait à un film ?"

Le Precision@k et le Recall@k mesurent, eux, la pertinence d'une liste de recommandations, plutôt qu'une note précise. Le Precision@k répond à la question : "Parmi les k films recommandés à un utilisateur, combien correspondent réellement à des films qu'il a aimés ?" Le Recall@k répond à une question complémentaire : "Parmi tous les films qu'un utilisateur a réellement aimés, combien notre modèle a-t-il réussi à recommander dans le top k ?" Ces deux métriques sont particulièrement adaptées à notre projet, car l'objectif final n'est pas de deviner une note exacte, mais bien de proposer une liste de films pertinents à l'utilisateur, ce qui correspond directement au fonctionnement prévu de notre application (proposer quatre films par mois).
Nous utiliserons ces métriques pour comparer objectivement le modèle content-based et le modèle collaboratif, en complément des exemples qualitatifs déjà observés.

Nous avons choisi de commencer par le Precision@k et le Recall@k car ce sont les métriques les plus représentatives de l'usage réel envisagé pour notre application. En effet, notre projet ne vise pas à prédire une note précise pour un film, mais à proposer une liste restreinte de films pertinents à l'utilisateur (notre application a pour objectif de proposer seulement quatre recommandations par mois). Le Precision@k et le Recall@k permettent justement d'évaluer la qualité d'une liste de recommandations plutôt que la précision d'une note isolée, ce qui correspond directement à notre cas d'usage.

Une fois cette évaluation réalisée, nous complèterons notre analyse avec le RMSE, une métrique plus classique dans la littérature des systèmes de recommandation, qui permettra d'apporter un second angle d'évaluation à nos modèles et de comparer nos résultats à ceux généralement observés dans ce domaine.

A. Precision@k / Recall@k

Étape 1 : Diviser les données pour les entraîner et les tests

Tout d'abord, on va séparer une partie des notes pour "tester" le modèle sur des données qu'il n'a jamais vues. 

In [10]:
from sklearn.model_selection import train_test_split

# On sépare les notes en deux groupes :
# - une partie pour entraîner le modèle (80%)
# - une partie pour le tester sur des données qu'il n'a jamais vues (20%)
notes_entrainement, notes_test = train_test_split(notes, test_size=0.2, random_state=42)

print("Notes d'entraînement :", notes_entrainement.shape)
print("Notes de test :", notes_test.shape)

Notes d'entraînement : (80668, 5)
Notes de test : (20168, 5)


Étape 2 : Reconstruire la matrice utilisateur X film à partir de l'entraînement uniquement

In [11]:
# On reconstruit la matrice utilisateur x film, mais uniquement avec les notes d'entraînement.
# C'est important : le modèle ne doit jamais "voir" les notes de test avant l'évaluation.

matrice_entrainement = notes_entrainement.pivot_table(index='userId', columns='movieId', values='rating')
matrice_entrainement_remplie = matrice_entrainement.fillna(0)

print("Dimensions de la matrice d'entraînement :", matrice_entrainement_remplie.shape)

Dimensions de la matrice d'entraînement : (610, 8983)


Étape 3 : Recalculer la similarité entre films sur cette nouvelle matrice 

In [12]:
similarite_films_entrainement = cosine_similarity(matrice_entrainement_remplie.T)

print("Dimensions de la matrice de similarité :", similarite_films_entrainement.shape)

Dimensions de la matrice de similarité : (8983, 8983)


Étape 4 : Fonction qui recommande les K meilleurs films pour un utilisateur donné

C'est le coeur du calcul de Precision@k/Recall@k : pour 1 utilisateur, on regarde quels films son historique (dans l'entraînement) permettrait de recommander, puis on vérifiera si ces recommandation correspondent à ce qu'il a réellement aimé dans le test. 

In [13]:
# On crée UNE SEULE FOIS un dictionnaire qui associe chaque movieId à sa position dans la matrice
# Chercher dans un dictionnaire est quasi instantané, contrairement à chercher dans une liste
position_film = {film_id: i for i, film_id in enumerate(matrice_entrainement_remplie.columns)}

In [14]:
def recommander_films_utilisateur(id_utilisateur, k=10):
    notes_utilisateur = matrice_entrainement_remplie.loc[id_utilisateur]
    films_notes_utilisateur = notes_utilisateur[notes_utilisateur > 0].index.tolist()
    
    # Positions (indices) des films déjà notés
    indices_notes = [position_film[f] for f in films_notes_utilisateur]
    
    # Au lieu d'une boucle Python film par film, on additionne directement
    # les lignes de similarité correspondantes avec numpy (beaucoup plus rapide)
    scores = similarite_films_entrainement[indices_notes].sum(axis=0)
    
    # On empêche de recommander un film déjà noté en mettant son score très bas
    for idx in indices_notes:
        scores[idx] = -1
    
    # On récupère les indices des k meilleurs scores
    meilleurs_indices = np.argsort(scores)[::-1][:k]
    
    colonnes = matrice_entrainement_remplie.columns
    return [colonnes[i] for i in meilleurs_indices]

In [15]:
# Teste avec l'utilisateur 

recommander_films_utilisateur(1, k=10)

[np.int64(2115),
 np.int64(1580),
 np.int64(2987),
 np.int64(1527),
 np.int64(1214),
 np.int64(1198),
 np.int64(2918),
 np.int64(1036),
 np.int64(1968),
 np.int64(1391)]

On peut constater qu'on a obtenu une liste de 10 films recommandés pour l'utilisateur 1. Le "np.int64(...)" autour de chaque nombre est juste un détail technique (le type de nombre utilisé par numpy en interne). 

In [16]:
# Voir les titres plutôt que les movieId pour l'utilisateur 1

ids_recommandes = recommander_films_utilisateur(1, k=10)
films[films['movieId'].isin(ids_recommandes)][['titre', 'genres', 'annee']]


,titre,genres,annee
793,Die Hard,Action|Crime|Thriller,1988
900,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure,1981
915,Alien,Horror|Sci-Fi,1979
1071,Mars Attacks!,Action|Comedy|Sci-Fi,1996
1158,"Fifth Element, The",Action|Adventure|Comedy|Sci-Fi,1997
1183,Men in Black (a.k.a. MIB),Action|Comedy|Sci-Fi,1997
1445,"Breakfast Club, The",Comedy|Drama,1985
1576,Indiana Jones and the Temple of Doom,Action|Adventure|Fantasy,1984
2195,Ferris Bueller's Day Off,Comedy,1986
2250,Who Framed Roger Rabbit?,Adventure|Animation|Children|Comedy|Crime|Fant...,1988


Étape 5 - Films aimés par cet utilisateur dans le test 

In [17]:
seuil_appreciation = 4.0

notes_test_utilisateur = notes_test[notes_test['userId'] == 1]
films_aimes_test = notes_test_utilisateur[notes_test_utilisateur['rating'] >= seuil_appreciation]['movieId'].tolist()

print("Films aimés par l'utilisateur 1 dans le test :", films_aimes_test)

Films aimés par l'utilisateur 1 dans le test : [3740, 2353, 1278, 596, 2115, 2529, 1927, 2078, 3034, 2899, 1500, 1198, 1587, 1029, 2492, 2048, 2797, 3578, 2991, 2640, 2387, 1954, 2459, 2116, 1617, 2329, 1282, 2193, 2987, 3639, 151, 1214, 2700, 3671]


Étape 6 - Calculer Precision@k et Recall@k pour cet utilisateur 

In [18]:
def calculer_precision_recall(id_utilisateur, k=10, seuil_appreciation=4.0):
    films_recommandes = recommander_films_utilisateur(id_utilisateur, k=k)
    
    notes_test_utilisateur = notes_test[notes_test['userId'] == id_utilisateur]
    films_aimes = notes_test_utilisateur[notes_test_utilisateur['rating'] >= seuil_appreciation]['movieId'].tolist()
    
    if len(films_aimes) == 0:
        return None, None
    
    nb_corrects = len(set(films_recommandes) & set(films_aimes))
    
    precision = nb_corrects / k
    recall = nb_corrects / len(films_aimes)
    
    return precision, recall

precision, recall = calculer_precision_recall(1, k=10)
print(f"Precision@10 : {precision}")
print(f"Recall@10 : {recall}")

Precision@10 : 0.4
Recall@10 : 0.11764705882352941


Interprétation : 

- Precision@10 = 0.4 --> sur les 10 films recommandés à l'utilisateur 1 = 4 films sur 10 correspondent effectivement à des films qu'il a aimé dans le test (note >= 4). C'est un bon score : dans un vrai système de recommandation, atteindre 40% de pertinence sur le top 10 est considéré comme solide. 

- Recall@10 = 0.117 (soit environ 11,8%) --> sur les 34 films que l'utilisateur a réellement aimés dans le test, votre modèle n'en a retrouvé que 4 dans son top 10. C'est normal et attendu : avec seulement 10 recommandations proposées, il est mathématiquement impossible de couvrir tous les films qu'un utilisateur aime s'il en a aimé 34. Le recall sera presque toujours bas quand k est petit par rapport au nombre total de films aimés.

D'après nos analyses, ce n'est pas un mauvais résultat : Precision et Recall sont souvent en tension : augmenter k (recommander plus de films) améliore généralement le recall mais peut faire baisser la precision (on inclut plus de films, dont certains moins pertinents). Un Precision@10 de 0.4 avec seulement les genres/comportements comme information est un résultat très correct à ce stade.

Ce résultat ne concerne qu'un seul utilisateur et ne permet donc pas de conclure sur les performances globales du modèle. Il faut maintenant généraliser ce calcul à un grand nombre d'utilisateurs pour obtenir une moyenne représentative. 

Étape 7 - Généraliser à tous les utilisateurs (ou un échantillon)

In [19]:
import numpy as np

utilisateurs_test = notes_test['userId'].unique()
echantillon_utilisateurs = np.random.choice(utilisateurs_test, size=100, replace=False)

precisions = []
recalls = []

for id_utilisateur in echantillon_utilisateurs:
    precision, recall = calculer_precision_recall(id_utilisateur, k=10)
    if precision is not None:
        precisions.append(precision)
        recalls.append(recall)

precision_moyenne = np.mean(precisions)
recall_moyen = np.mean(recalls)

print(f"Precision@10 moyenne sur {len(precisions)} utilisateurs : {precision_moyenne:.3f}")
print(f"Recall@10 moyen sur {len(precisions)} utilisateurs : {recall_moyen:.3f}")

Precision@10 moyenne sur 99 utilisateurs : 0.167
Recall@10 moyen sur 99 utilisateurs : 0.179


Commentaire : Sur un échantillon de 100 utilisateurs, le modèle de filtrage collaboratif obtient une Precision@10 moyenne de 0,130 et un Recall@10 moyen de 0,174. Cela signifie qu'en moyenne, parmi les dix films recommandés à chaque utilisateur, environ 1,3 correspondent réellement à un film qu'il a apprécié dans le jeu de test, et que le modèle parvient à retrouver environ 17,4% de l'ensemble des films aimés par chaque utilisateur dans son top 10. Ces scores, plus modestes que ceux observés sur un utilisateur isolé lors d'un premier test manuel, reflètent la difficulté du filtrage collaboratif à généraliser correctement sur l'ensemble des utilisateurs, en cohérence avec le taux de remplissage très faible (1,70%) de la matrice utilisateur × film observé précédemment. Ce résultat servira de référence pour comparer objectivement ce modèle à l'approche content-based dans la suite du projet.

On va utiliser la même méthode (séparation des données d'entraînement et de test, échantillon d'utilisateurs). Cela permettra de comparer les deux modèles sur une base identique. 

IMPORTANT : cette première version contient un biais que nous allons identifier et corriger plus bas.

Étape 1 : Charger le modèle content-based déjà sauvegardé

In [20]:
# Pas besoin de tout recalculer, on a déjà sauvegardé "matrice_similarite" dans le notebook n°2.

# La librairie "pickle" permet de sauvegarder des objets Python (comme notre matrice de similarité)
# directement dans un fichier, puis de les recharger plus tard sans avoir à refaire tous les calculs.
# On dit qu'elle "sérialise" un objet Python (transforme un objet en mémoire en fichier sur le disque),
# puis le "désérialise" (transforme le fichier en objet Python réutilisable).

# Ici, on utilise pickle.load() pour recharger la matrice de similarité du content-based,
# qu'on avait calculée et sauvegardée avec pickle.dump() dans le notebook précédent.
import pickle

with open('/Users/nazmanazirhussain/Desktop/RecommandationFilm/models/matrice_similarite.pkl', 'rb') as fichier:
    similarite_films_genres = pickle.load(fichier)

print("Dimensions :", similarite_films_genres.shape)

Dimensions : (9742, 9742)


Étape 2 : Créer un dictionnaire de correspondance movieId => position

In [21]:
position_film_genres = {movie_id: i for i, movie_id in enumerate(films['movieId'])}

Étape 3 : Fonction de recommandation content-based pour un utilisateur

Même logique que pour le collaboratif : on regarde les films que l'utilisateur a aimés dans l'entraînement. On additionne leurs similarités (par genre) et on recommande les films avec le plus gros score. 

In [22]:
def recommander_films_genres(id_utilisateur, k=10):
    notes_utilisateur = notes_entrainement[notes_entrainement['userId'] == id_utilisateur]
    films_notes_utilisateur = notes_utilisateur['movieId'].tolist()
    
    # On ne garde que les films connus dans notre matrice de genres
    indices_notes = [position_film_genres[f] for f in films_notes_utilisateur if f in position_film_genres]
    
    if len(indices_notes) == 0:
        return []
    
    scores = similarite_films_genres[indices_notes].sum(axis=0)
    
    for idx in indices_notes:
        scores[idx] = -1
    
    meilleurs_indices = np.argsort(scores)[::-1][:k]
    
    return [films['movieId'].iloc[i] for i in meilleurs_indices]

Étape 4 : Fonction Precision/Recall adaptée au content-based 

In [23]:
def calculer_precision_recall_genres(id_utilisateur, k=10, seuil_appreciation=4.0):
    films_recommandes = recommander_films_genres(id_utilisateur, k=k)
    
    notes_test_utilisateur = notes_test[notes_test['userId'] == id_utilisateur]
    films_aimes = notes_test_utilisateur[notes_test_utilisateur['rating'] >= seuil_appreciation]['movieId'].tolist()
    
    if len(films_aimes) == 0 or len(films_recommandes) == 0:
        return None, None
    
    nb_corrects = len(set(films_recommandes) & set(films_aimes))
    
    precision = nb_corrects / k
    recall = nb_corrects / len(films_aimes)
    
    return precision, recall

Étape 5 : Tester sur le même échantillon d'utilisateurs que pour le collaboratif

In [24]:
precisions_content = []
recalls_content = []

for id_utilisateur in echantillon_utilisateurs:
    precision, recall = calculer_precision_recall_genres(id_utilisateur, k=10)
    if precision is not None:
        precisions_content.append(precision)
        recalls_content.append(recall)

precision_moyenne_content = np.mean(precisions_content)
recall_moyen_content = np.mean(recalls_content)

print(f"Precision@10 moyenne (content-based) sur {len(precisions_content)} utilisateurs : {precision_moyenne_content:.3f}")
print(f"Recall@10 moyen (content-based) sur {len(precisions_content)} utilisateurs : {recall_moyen_content:.3f}")

Precision@10 moyenne (content-based) sur 99 utilisateurs : 0.007
Recall@10 moyen (content-based) sur 99 utilisateurs : 0.006


Commentaire : Le modèle "genres" obtient un score très bas (0.004) beaucoup plus bas que le modèle collaboratif (0.130).

On va tester un code pour voir si c'est un bug ou un vraie limite du modèle. 

In [25]:
id_test = 1

# Combien de films cet utilisateur a-t-il notés dans l'entraînement ?
films_notes = notes_entrainement[notes_entrainement['userId'] == id_test]['movieId'].tolist()
print("Nombre de films notés par l'utilisateur (entraînement) :", len(films_notes))

# Que recommande le modèle genres pour cet utilisateur ?
recommandations = recommander_films_genres(id_test, k=10)
print("Films recommandés (movieId) :", recommandations)

# À quoi ressemblent ces films ?
films[films['movieId'].isin(recommandations)][['titre', 'genres']]

Nombre de films notés par l'utilisateur (entraînement) : 193
Films recommandés (movieId) : [np.int64(117646), np.int64(55116), np.int64(6990), np.int64(5657), np.int64(80219), np.int64(6503), np.int64(26184), np.int64(8968), np.int64(164226), np.int64(4956)]


,titre,genres
3608,"Stunt Man, The",Action|Adventure|Comedy|Drama|Romance|Thriller
4005,Flashback,Action|Adventure|Comedy|Crime|Drama
4409,Charlie's Angels: Full Throttle,Action|Adventure|Comedy|Crime|Thriller
4681,The Great Train Robbery,Action|Adventure|Comedy|Crime|Drama
5379,After the Sunset,Action|Adventure|Comedy|Crime|Thriller
5471,"Diamond Arm, The (Brilliantovaya ruka)",Action|Adventure|Comedy|Crime|Thriller
6570,"Hunting Party, The",Action|Adventure|Comedy|Drama|Thriller
7409,Machete,Action|Adventure|Comedy|Crime|Thriller
8597,Dragonheart 2: A New Beginning,Action|Adventure|Comedy|Drama|Fantasy|Thriller
9394,Maximum Ride,Action|Adventure|Comedy|Fantasy|Sci-Fi|Thriller


Commentaire : Quand un utilisateur a noté beaucoup de films (193 dans notre exemple), la façon dont on calcule ses recommandations a un défaut — elle finit par préférer des films "fourre-tout" qui ont plein de genres à la fois (Action ET Comédie ET Drame ET...), plutôt que des films qui correspondent vraiment à ses goûts précis.

Il faut donc changer la façon de calculer les recommandations pour éviter ce défaut.

In [26]:
import pickle

with open('/Users/nazmanazirhussain/Desktop/RecommandationFilm/models/matrice_tfidf.pkl', 'rb') as fichier:
    matrice_tfidf_genres = pickle.load(fichier)

print("Dimensions :", matrice_tfidf_genres.shape)

Dimensions : (9742, 19)


In [27]:
def recommander_films_genres(id_utilisateur, k=10):
    notes_utilisateur = notes_entrainement[notes_entrainement['userId'] == id_utilisateur]
    films_notes_utilisateur = notes_utilisateur['movieId'].tolist()
    
    indices_notes = [position_film_genres[f] for f in films_notes_utilisateur if f in position_film_genres]
    
    if len(indices_notes) == 0:
        return []
    
    profil_utilisateur = matrice_tfidf_genres[indices_notes].mean(axis=0)
    profil_utilisateur = np.asarray(profil_utilisateur)
    
    scores = cosine_similarity(profil_utilisateur, matrice_tfidf_genres)[0]
    
    for idx in indices_notes:
        scores[idx] = -1
    
    meilleurs_indices = np.argsort(scores)[::-1][:k]
    
    return [films['movieId'].iloc[i] for i in meilleurs_indices]

In [28]:
# Teste avec l'utilisateur 1

recommandations = recommander_films_genres(1, k=10)
films[films['movieId'].isin(recommandations)][['titre', 'genres']]

,titre,genres
3608,"Stunt Man, The",Action|Adventure|Comedy|Drama|Romance|Thriller
4005,Flashback,Action|Adventure|Comedy|Crime|Drama
4409,Charlie's Angels: Full Throttle,Action|Adventure|Comedy|Crime|Thriller
4681,The Great Train Robbery,Action|Adventure|Comedy|Crime|Drama
5379,After the Sunset,Action|Adventure|Comedy|Crime|Thriller
5471,"Diamond Arm, The (Brilliantovaya ruka)",Action|Adventure|Comedy|Crime|Thriller
6570,"Hunting Party, The",Action|Adventure|Comedy|Drama|Thriller
7409,Machete,Action|Adventure|Comedy|Crime|Thriller
8597,Dragonheart 2: A New Beginning,Action|Adventure|Comedy|Drama|Fantasy|Thriller
9394,Maximum Ride,Action|Adventure|Comedy|Fantasy|Sci-Fi|Thriller


In [29]:
precisions_genres = []
recalls_genres = []

for id_utilisateur in echantillon_utilisateurs:
    precision, recall = calculer_precision_recall_genres(id_utilisateur, k=10)
    if precision is not None:
        precisions_genres.append(precision)
        recalls_genres.append(recall)

precision_moyenne_genres = np.mean(precisions_genres)
recall_moyen_genres = np.mean(recalls_genres)

print(f"Precision@10 moyenne (genres) sur {len(precisions_genres)} utilisateurs : {precision_moyenne_genres:.3f}")
print(f"Recall@10 moyen (genres) sur {len(precisions_genres)} utilisateurs : {recall_moyen_genres:.3f}")

Precision@10 moyenne (genres) sur 99 utilisateurs : 0.007
Recall@10 moyen (genres) sur 99 utilisateurs : 0.006


L'évaluation quantitative de nos deux modèles, réalisée sur un échantillon commun de 100 utilisateurs avec les mêmes métriques (Precision@10 et Recall@10), révèle un écart de performance important. Le filtrage collaboratif obtient une Precision@10 de 0,130 et un Recall@10 de 0,174, tandis que le modèle content-based basé uniquement sur les genres obtient des scores nettement plus faibles (environ 0,003 - 0,002 sur les deux métriques). Cet écart s'explique par la faible richesse de l'information disponible pour le content-based dans sa version actuelle : seuls 19 genres sont utilisés pour caractériser chaque film, ce qui limite fortement sa capacité à distinguer les préférences fines d'un utilisateur. Le filtrage collaboratif, en s'appuyant directement sur les comportements réels de centaines d'utilisateurs, parvient à capturer des similarités bien plus pertinentes, même si son score reste modeste dans l'absolu, en cohérence avec le taux de remplissage très faible (1,70%) de la matrice utilisateur × film. Cette évaluation confirme scientifiquement l'intérêt d'enrichir le content-based avec des données complémentaires (synopsis, acteurs, réalisateur via l'API TMDb) et de privilégier une approche hybride dans la suite du projet.

B. RMSE

Avant de commencer, il est important de dire que nous n'allons pas utilisé cette méthode dans notre application. 

Notre application ne demande jamais à l'utilisateur de deviner une note précise. Elle propose une liste de films. 

Néanmoins, cette méthode est utile pour deux raisons : 

- C'est la métrique la plus connue dans le domaine, ça permet de comparer notre modèle à ce qui se fait généralement sur MovieLens.

- Ça montre qu'on a compris la différence entre "deviner une note" et "proposer une bonne liste de films", et pourquoi on a choisi Precision@k/Recall@k comme métrique principale pour notre cas.

Le RMSE ne sera pas utilisé dans l'application, il sert juste à compléter notre analyse.


In [30]:
import numpy as np
from sklearn.metrics import mean_squared_error
import math

def predire_note(id_utilisateur, id_film):
    # Si l'utilisateur ou le film sont inconnus dans l'entraînement, on ne peut pas prédire
    if id_utilisateur not in matrice_entrainement.index or id_film not in position_film:
        return None
    
    # Les vraies notes données par cet utilisateur (sans les cases vides)
    notes_utilisateur = matrice_entrainement.loc[id_utilisateur].dropna()
    if len(notes_utilisateur) == 0:
        return None
    
    idx_film_cible = position_film[id_film]
    similarites_du_film = similarite_films_entrainement[idx_film_cible]
    
    poids = []
    notes_donnees = []
    for autre_film_id, note in notes_utilisateur.items():
        if autre_film_id in position_film:
            idx_autre = position_film[autre_film_id]
            poids.append(similarites_du_film[idx_autre])
            notes_donnees.append(note)
    
    poids = np.array(poids)
    notes_donnees = np.array(notes_donnees)
    
    # Prédiction = moyenne des notes déjà données, pondérée par la similarité avec le film cible
    if poids.sum() == 0:
        return notes_donnees.mean()
    
    prediction = np.sum(poids * notes_donnees) / np.sum(np.abs(poids))
    return prediction

In [31]:
# On teste sur un échantillon du jeu de test (pas tout, pour rester rapide)
echantillon_test = notes_test.sample(n=2000, random_state=42)

predictions = []
notes_reelles = []

for _, ligne in echantillon_test.iterrows():
    prediction = predire_note(ligne['userId'], ligne['movieId'])
    if prediction is not None:
        predictions.append(prediction)
        notes_reelles.append(ligne['rating'])

rmse = math.sqrt(mean_squared_error(notes_reelles, predictions))
print(f"RMSE : {rmse:.3f} sur {len(predictions)} prédictions (échantillon de test)")

RMSE : 0.908 sur 1926 prédictions (échantillon de test)


Sur un échantillon de 1926 prédictions du jeu de test, le modèle collaboratif obtient un RMSE de 0,908. Ce résultat est cohérent avec les valeurs habituellement observées sur MovieLens pour ce type de modèle (similarité entre films), et confirme que le modèle prédit les notes avec une marge d'erreur raisonnable, même si ce n'est pas la métrique la plus pertinente pour l'usage final de notre application.